In [11]:
# ========================================
# 🧠 TASK 6 - Semantic Video Search API
# ========================================

# Install required package
!pip install flask

In [12]:
# ========================================
# 📦 Import Libraries
# ========================================
from flask import Flask, request, jsonify
from sentence_transformers import SentenceTransformer, util
import torch
import json
import pandas as pd

In [13]:
# ========================================
# 🧩 VideoSearchEngine Class (from Task 5)
# ========================================

class VideoSearchEngine:
    def __init__(self, data_path="videos.json", model_name="all-MiniLM-L6-v2"):
        # Load your pretrained SentenceTransformer model
        self.model = SentenceTransformer(model_name)
        
        # Load the video dataset (You can replace this with your own JSON)
        with open(data_path, "r", encoding="utf-8") as f:
            self.videos = json.load(f)
        
        # Encode all video titles/descriptions
        self.video_texts = [video["title"] + " " + video["description"] for video in self.videos]
        self.embeddings = self.model.encode(self.video_texts, convert_to_tensor=True)

    def search(self, query, top_k=5):
        query_embedding = self.model.encode(query, convert_to_tensor=True)
        scores = util.pytorch_cos_sim(query_embedding, self.embeddings)[0]
        top_results = torch.topk(scores, k=top_k)
        
        results = []
        for idx, score in zip(top_results.indices, top_results.values):
            video = self.videos[idx]
            results.append({
                "title": video["title"],
                "description": video["description"],
                "url": video.get("url", ""),
                "score": round(float(score), 4)
            })
        return results


In [14]:
# ========================================
# 🧰 Sample Data (for Testing)
# ========================================

sample_data = [
    {"title": "Learn Python Basics", "description": "Introduction to Python programming for beginners.", "url": "https://youtu.be/python1"},
    {"title": "Machine Learning Tutorial", "description": "Comprehensive guide to ML concepts.", "url": "https://youtu.be/ml1"},
    {"title": "Deep Learning with PyTorch", "description": "Step-by-step neural network building using PyTorch.", "url": "https://youtu.be/dl1"},
    {"title": "Flask API Development", "description": "How to build APIs with Flask framework.", "url": "https://youtu.be/flask1"},
    {"title": "FastAPI Crash Course", "description": "Build high-performance APIs using FastAPI.", "url": "https://youtu.be/fastapi1"}
]

# Save to file for VideoSearchEngine to read
with open("videos.json", "w", encoding="utf-8") as f:
    json.dump(sample_data, f, indent=4)


In [17]:
from flask import Flask, request, jsonify, send_from_directory
from flask_cors import CORS
from threading import Thread

app = Flask(__name__, static_folder='.')
CORS(app)

@app.route('/')
def home():
    return send_from_directory('.', 'index.html')

@app.route('/search', methods=['POST'])
def search_videos():
    try:
        data = request.get_json(force=True)
        query = data.get("query", "")
        top_k = int(data.get("top_k", 5))
        if not query:
            return jsonify({"error": "Query is required"}), 400

        results = [
            {"title": "Demo Video 1", "description": "Sample result", "url": "https://youtube.com", "score": 0.95},
            {"title": "Demo Video 2", "description": "Another result", "url": "https://youtube.com", "score": 0.90}
        ]
        return jsonify({"query": query, "results": results})
    except Exception as e:
        return jsonify({"error": str(e)}), 500

def run_app():
    app.run(debug=True, use_reloader=False)

Thread(target=run_app).start()


 * Serving Flask app '__main__'


 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
